# Study 10: Paper Report Generation\n**Goal:** Generate LaTeX tables, clinical summary, and evaluation report for the research paper.

## 1. Setup

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')
print('Setup complete')

## 2. Load Evaluation Results

In [ ]:
res_dir = Path.cwd().parent / 'results'
# Load all result files
ensemble = json.load(open(res_dir / 'ensemble_evaluation.json'))
stats = json.load(open(res_dir / 'statistical_tests.json'))
per_sample = json.load(open(res_dir / 'per_sample_dice.json'))
canonical = json.load(open(Path.cwd().parent / 'data' / 'metadata' / 'statistics.json'))

print(f'Loaded {len(ensemble)} model results')
print(f'Loaded {len(per_sample)} per-sample series')
print(f'Loaded {len(stats)} statistical comparisons')
print(f'Canonical stats: {len(canonical)} fields')

## 3. Summary Statistics Table

In [ ]:
rows = []
for name, metrics in ensemble.items():
    row = {'Model': name}
    for met, vals in metrics.items():
        row[f'{met.upper()}_mean'] = f'{vals["mean"]:.4f}'
        row[f'{met.upper()}_std'] = f'{vals["std"]:.4f}'
    rows.append(row)
df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())

# Identify best model by mean dice
best_model = max(ensemble, key=lambda n: ensemble[n]['dice']['mean'])
print(f'\nBest model: {best_model} (Dice={ensemble[best_model]["dice"]["mean"]:.4f})')

## 4. Statistical Significance Table

In [ ]:
sig_rows = []
for comp, res in stats.items():
    sig_rows.append({
        'Comparison': comp.replace('_vs_', ' vs '),
        'Dice Diff': f'{res["diff_mean"]:.4f}\u00b1{res["diff_std"]:.4f}',
        'p-value': f'{res["p_value"]:.4f}',
        'Significant': 'Yes' if res['significant'] else 'No',
    })
sig_df = pd.DataFrame(sig_rows)
print(sig_df.to_string(index=False))

## 5. LaTeX Table Generation

In [ ]:
def fmt_mean_std(mean, std):
    return f'${mean:.3f}\u00b1{std:.3f}$'

lines = [
    '% --- Auto-generated LaTeX Table ---',
    r'\begin{table}[h]',
    r'\centering',
    r'\caption{Quantitative comparison of segmentation models on the test set. '
    r'Values are mean $\pm$ standard deviation across all test slices.}',
    r'\label{tab:results}',
    r'\begin{tabular}{lccccc}',
    r'\toprule',
    r'Model & Dice $\uparrow$ & IoU $\uparrow$ & HD95 $\downarrow$ & ASD $\downarrow$ & NSD $\uparrow$ \\',
    r'\midrule',
]
for name in ['best_model', 'member_0', 'member_1', 'member_2', 'ensemble']:
    if name in ensemble:
        m = ensemble[name]
        row = f'{name} & {fmt_mean_std(m["dice"]["mean"], m["dice"]["std"])} & '
        row += f'{fmt_mean_std(m["iou"]["mean"], m["iou"]["std"])} & '
        row += f'{m["hd95"]["mean"]:.1f}$\pm${m["hd95"]["std"]:.1f} & '
        row += f'{m["asd"]["mean"]:.2f}$\pm${m["asd"]["std"]:.2f} & '
        row += f'{fmt_mean_std(m["nsd"]["mean"], m["nsd"]["std"])} \\\\'
        lines.append('    ' + row)
lines += [
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table}',
]
tex = '\n'.join(lines)
with open(res_dir / 'latex_table.tex', 'w') as f:
    f.write(tex)
print(f'LaTeX table saved to {res_dir / "latex_table.tex"}')
print(tex)

## 6. Clinical Summary

In [ ]:
clinical_lines = [
    '% --- Clinical Summary ---',
    r'\section*{Clinical Summary}',
    f'\\textbf{{Dataset}}: {canonical.get("n_total_slices", "N/A")} 2D CT slices '
    f'from {canonical.get("n_volumes", "N/A")} liver tumor volumes.',
    f'\\textbf{{Class imbalance}}: {canonical.get("imbalance_ratio", "N/A"):.0f}:1 background to tumor.',
    f'\\textbf{{Best model}}: {best_model} with Dice={ensemble[best_model]["dice"]["mean"]:.4f}.',
    f'\\textbf{{Ensemble improvement}}: {stats.get(f"{best_model}_vs_{best_model}", {}).get("diff_mean", 0):.4f} Dice '
        f'($p={stats.get(f"{best_model}_vs_{best_model}", {}).get("p_value", "N/A"):.4f}$).',
    f'\\textbf{{Date}}: {datetime.now().strftime("%Y-%m-%d")}.',
]
with open(res_dir / 'clinical_summary.tex', 'w') as f:
    f.write('\n'.join(clinical_lines))
print(f'Saved clinical summary to {res_dir / "clinical_summary.tex"}')
print('\n'.join(clinical_lines))

## 7. Full Evaluation Report

In [ ]:
report = '# Liver Tumor Segmentation - Evaluation Report\n\n'report += '## Dataset Statistics\n'n_slices = canonical.get('n_total_slices', 'N/A')n_vols = canonical.get('n_volumes', 'N/A')tumor_pct = canonical.get('tumor_slice_pct', 'N/A')imbal = canonical.get('imbalance_ratio', 'N/A')report += f'- Total slices: {n_slices}\n'report += f'- Volumes: {n_vols}\n'report += f'- Tumor slice ratio: {tumor_pct:.2f}%\n'report += f'- Imbalance: {imbal:.0f}:1\n\n'report += '## Model Performance\n\n'report += '| Model | Dice | IoU | HD95 | ASD | NSD |\n'report += '|-------|------|-----|------|-----|-----|\n'for name in ['best_model', 'member_0', 'member_1', 'member_2', 'ensemble']:    if name in ensemble:        m = ensemble[name]        report += f'| {name} | {m["dice"]["mean"]:.4f}+-{m["dice"]["std"]:.4f} |'        report += f' {m["iou"]["mean"]:.4f}+-{m["iou"]["std"]:.4f} |'        report += f' {m["hd95"]["mean"]:.1f}+-{m["hd95"]["std"]:.1f} |'        report += f' {m["asd"]["mean"]:.2f}+-{m["asd"]["std"]:.2f} |'        report += f' {m["nsd"]["mean"]:.4f}+-{m["nsd"]["std"]:.4f} |\n'report += '\n## Statistical Significance\n\n'for comp, res in stats.items():    report += f'- {comp}: diff={res["diff_mean"]:.4f}, p={res["p_value"]:.4f}\n'with open(res_dir / 'evaluation_report.md', 'w') as f:    f.write(report)print(f'Saved report to {res_dir / "evaluation_report.md"}')print(report[:500])

## 8. Generate All Outputs

In [ ]:
print('Paper report files generated:')
for f in ['latex_table.tex', 'clinical_summary.tex', 'evaluation_report.md']:
    path = res_dir / f
    if path.exists(): print(f'  {path} ({path.stat().st_size} bytes)')
print('\nDone.')